# 2. Measurement of $B_s^0 \to \mu^+ \mu^-$ branching ratio and search for the $B_d^0 \to \mu^+ \mu^-$. 

The following tutorial will teach you how the branchin ratio measurements and searcher for decays (bump hunts) are performed using a simple toy dataset based on the results from the https://linkinghub.elsevier.com/retrieve/pii/S0370269323002897. 

You are given 7 files: 

+ bs2mumu_toy.root contains the toy simulation of the $B_s^0 \to \mu^+ \mu^-$ mass range.
+ peaking_bkg_toy.root contains the toy simulation of the peaking backgrounds.
+ semi_bkg_toy.root containe the toy simulation of the semileptonic backgrounds. 
+ full_toy.root contains the toy simulation of the dimuon spectra near the $B_s^0$ mass range.
+ norm_channel_toy.root contains the toy data
+ bu2jpsikplus_toy.root contains the toy of the $B^+ \to J/\psi K^+$
+ peaking_bkg_bu2jspikplus_toy.root contains toy of the peaking background. 

All toys assume full selection is applied from the https://linkinghub.elsevier.com/retrieve/pii/S0370269323002897. 
The original analysis had split the dataset into $2_{eta} \times 2_{bdt} \times 4_{era}$ categories with two categories for different geometrical part of detector, two for different bdt regions and four for different data-taking periods. 
Here we simplify things into one category. 

You will be using [ROOT](https://root.cern/) package in order to do all the questions and [RooFit](https://root.cern/manual/roofit/) to fit.

2.a : Write a list of normalization channels that might be used for the measurement of the $B_s^0 \to \mu^+ \mu^-$ BR ? What criteria are improtant for choosing the normalization channel? Use PDG for help. 

2.b : We will stick to the analysis defined normalization channel $B^+ \to J/\psi K^+$. Why what this decay chosen? What are pros and consof this decay mode? Write down the master formula for the BR of $B_s^0 \to \mu^+ \mu^-$ with $B^+ \to J/\psi K^+$ are normalization channel.

2.c : Measure the yield of the normalization channel in the toy data. There are three files to help you with that: 

+ norm_channel_toy.root contains the toy data
+ bu2jpsikplus_toy.root contains the toy of the $B^+ \to J/\psi K^+$
+ peaking_bkg_bu2jspikplus_toy.root contains toy of the peaking background. 

Use the bu2jpsikplus_toy.root and peaking_bkg_bu2jspikplus_toy.root to find the shape of the $B^+ \to J/\psi K^+$ signal peak and the shape of the peaking background. Use this information to fit the norm_channel_toy.root with the extended maximum likelihood unbinned fit.

In [11]:
import ROOT
ROOT.gStyle.SetOptStat(0)

file_in_full = ROOT.TFile.Open("norm_channel_toy.root", "readonly")
norm_full = file_in_full.Get("Events")
file_in_shape  = ROOT.TFile.Open("bu2jpsikplus_toy.root", "readonly")
norm_shape = file_in_shape.Get("signal_tree")
peaking_filein = ROOT.TFile.Open("peaking_bkg_bu2jspikplus_toy.root", "readonly")
peaking_shape = peaking_filein.Get("peak_tree")

In [12]:


nbins = 80
xmin  = 5.1
xmax  = 5.45

h_norm  = ROOT.TH1D("h_norm","",nbins,xmin,xmax)
h_sig   = ROOT.TH1D("h_sig",";m(J/#psi K) [GeV];Normalized entries",nbins,xmin,xmax)
h_peak  = ROOT.TH1D("h_peak","",nbins,xmin,xmax)

norm_full.Draw("mass>>h_norm","","goff")
norm_shape.Draw("mass>>h_sig","","goff")
peaking_shape.Draw("mass>>h_peak","","goff")

h_norm.SetLineColor(ROOT.kBlack)
h_norm.SetLineWidth(2)
h_norm.Scale(1.0/h_norm.Integral())

h_sig.SetLineColor(ROOT.kBlue)
h_sig.SetLineWidth(2)
h_sig.Scale(1.0 / h_sig.Integral()) 

h_peak.SetLineColor(ROOT.kRed)
h_peak.SetLineWidth(2)
h_peak.Scale(1.0/h_peak.Integral())

c = ROOT.TCanvas("c","Normalization channel shapes",900,700)


h_sig.Draw()
h_peak.Draw("same")
h_norm.Draw("same")

leg = ROOT.TLegend(0.65,0.70,0.88,0.88)
leg.AddEntry(h_norm,"Full toy sample","l")
leg.AddEntry(h_sig,"B^{+} #rightarrow J/#psi K^{+} signal","l")
leg.AddEntry(h_peak,"Peaking background","l")
leg.Draw()

c.Draw()
c.SaveAs("normalization_shapes_histograms.pdf")

Info in <TCanvas::Print>: pdf file normalization_shapes_histograms.pdf has been created


You are given the fitting skeleton - execute the following cells 

In [13]:
mass = ROOT.RooRealVar("mass", r"$m(J/#psi K^{+})$", 5.1, 5.45) #define observable

file_in_shape  = ROOT.TFile.Open("/eos/user/v/valukash/flavour-course/exercises/bu2jpsikplus_toy.root", "readonly")
norm_shape = file_in_shape.Get("signal_tree")
#Define the dataset for fitting
dataset = ROOT.RooDataSet("data", "", ROOT.RooArgList(mass), ROOT.RooFit.Import(norm_shape))
dataset.Print("v")
#Number of the B+ decays. In the context of the simulated signal toy it is not intersting (RooFit normalizes automatically), but it is crucial for the full fit.

N_bu = ROOT.RooRealVar("N_bu", "", 0.9 * dataset.numEntries(), 0., 1.2 * dataset.numEntries())

#define shape parameters double-gauss + CB
mu    = ROOT.RooRealVar("mu","mu",5.2779, 5.0, 5.5)
sigma = ROOT.RooRealVar("sigma","sigma",0.012, 0., 0.5)

f1    = ROOT.RooRealVar("f1","f1",0.093, 0., 1.)
SG1   = ROOT.RooRealVar("SG1","SG1",4.616, 2.0, 6.0)
f     = ROOT.RooRealVar("f","f",0.421, 0., 1.)
SCB   = ROOT.RooRealVar("SCB","SCB",1.740, 0.5, 2.5)
alpha = ROOT.RooRealVar("alpha","alpha",2.261, 0., 5.)
n     = ROOT.RooRealVar("n","n",2.444, 0., 5.)

sigma_cb = ROOT.RooFormulaVar("sigma_cb","@0*@1",ROOT.RooArgList(sigma,SCB))
sigma_g1 = ROOT.RooFormulaVar("sigma_g1","@0*@1",ROOT.RooArgList(sigma,SG1))


cb = ROOT.RooCBShape(
    "cb","CrystalBall",
    mass,mu,sigma_cb,alpha,n
)

gauss1 = ROOT.RooGaussian(
    "gauss1","gauss1",
    mass,mu,sigma_g1
)

gauss2 = ROOT.RooGaussian(
    "gauss2","gauss2",
    mass,mu,sigma
)

gauss12 = ROOT.RooAddPdf(
    "gauss12","double gaussian",
    ROOT.RooArgList(gauss1,gauss2),
    ROOT.RooArgList(f1)
)

signal_pdf_norm = ROOT.RooAddPdf(
    "signal_pdf_norm","signal",
    ROOT.RooArgList(cb,gauss12),
    ROOT.RooArgList(f)
)



fit_result = signal_pdf_norm.fitTo(dataset, 
                                 ROOT.RooFit.Minimizer("Minuit2"),
                                 ROOT.RooFit.Optimize(True), #optimize the treatment of constants in logL
                                 ROOT.RooFit.Offset(True), #set initial logL to 0
                                 ROOT.RooFit.Strategy(2), #internal code for the most precise minimization strategy of minuit2
                                 ROOT.RooFit.Hesse(True), #use Hesse to compute the uncertainty
                                 ROOT.RooFit.Save(True))

fit_result.Print("v")

c1 = ROOT.TCanvas("c1","fit",800,600)
frame = mass.frame()
dataset.plotOn(frame)
signal_pdf_norm.plotOn(frame)
frame.Draw()
c1.Update()
c1.SaveAs("normalization_signal_fit.pdf")
c1.Draw()

[#1] INFO:DataHandling -- RooAbsReal::attachToTree(mass) TTree Float_t branch mass will be converted to double precision.
DataStore data ()
  Contains 5000000 entries
  Observables: 
    1)  mass = 5.29484  L(5.1 - 5.45)  "$m(J/#psi K^{+})$"
[#0] WARNING:InputArguments -- The parameter 'sigma' with range [0, 0.5] of the RooGaussian 'gauss2' exceeds the safe range of (0, inf). Advise to limit its range.
[#1] INFO:Fitting -- RooAbsPdf::fitTo(signal_pdf_norm) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- Creation of NLL object took 78.515 ms
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_signal_pdf_norm_data) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
Minuit2Minimizer: Minimize with max-calls 4000 convergence for edm < 1 strate

Info in <Minuit2>: MnSeedGenerator Computing seed using NumericalGradient calculator
Info in <Minuit2>: MnSeedGenerator Evaluated function and gradient in 8.73928 s
Info in <Minuit2>: MnSeedGenerator Initial state: FCN =                 0 Edm =      0.7758556113 NCalls =     39
Info in <Minuit2>: MnHesse Done after 16.2187 s
Info in <Minuit2>: MnSeedGenerator run Hesse - Initial seeding state: 
  Minimum value : 0
  Edm           : 4.100980582
  Internal parameters:	[      0.242365851     0.3130901197   -0.09574622264    -0.1586648819    -0.9510055416     0.1118329628   -0.02240187366     -1.259704725]	
  Internal gradient  :	[      18.81039167      94.49155449      105.2053335      315.3679714     -753.0667579      21439.64371     -0.577095926      402.6363823]	
  Internal covariance matrix:
[[  2.4289468e-05  2.7820039e-05 -1.1592513e-05 -2.2260983e-05 -2.1640888e-05 -2.6098198e-09  4.6705171e-05  5.0319782e-07]
 [  2.7820039e-05  0.00019290995  -3.148564e-05  6.4275571e-05 -1.532787

The sample contains a small subset of the peaking background decays. Given that CMS has no particle identification capabilitie, what decays could compose peaking background to the normalization channel? 

In [14]:
#Now let's repeat it for the peaking background
#Define the dataset for fitting
peaking_filein = ROOT.TFile.Open("/eos/user/v/valukash/flavour-course/exercises/peaking_bkg_bu2jspikplus_toy.root", "readonly")
peaking_shape = peaking_filein.Get("peak_tree")
dataset2 = ROOT.RooDataSet("data2", "", ROOT.RooArgList(mass), ROOT.RooFit.Import(peaking_shape))

N_peak = ROOT.RooRealVar("N_peak", "", 0.9 * dataset2.numEntries(), 0., 1.2 * dataset2.numEntries())

mu_peak = ROOT.RooRealVar("mu_peak","mu_peak",5.22, 5.0, 5.5)

s1 = ROOT.RooRealVar("s1","s1",0.02, 0., 0.5)
s2 = ROOT.RooRealVar("s2","s2",0.03, 0., 0.5)
s3 = ROOT.RooRealVar("s3","s3",0.04, 0., 0.5)

g1 = ROOT.RooGaussian("g1","g1",mass,mu_peak,s1)
g2 = ROOT.RooGaussian("g2","g2",mass,mu_peak,s2)
g3 = ROOT.RooGaussian("g3","g3",mass,mu_peak,s3)

f12 = ROOT.RooRealVar("f12","f12",0.5, 0., 1.)
f23 = ROOT.RooRealVar("f23","f23",0.3, 0., 1.)

g12 = ROOT.RooAddPdf("g12","g12",ROOT.RooArgList(g1,g2),ROOT.RooArgList(f12))
peak_pdf_norm = ROOT.RooAddPdf("peak_pdf_norm","peak",
                          ROOT.RooArgList(g12,g3),
                          ROOT.RooArgList(f23))

fit_result = peak_pdf_norm.fitTo(dataset2, 
                                 ROOT.RooFit.Minimizer("Minuit2"),
                                 ROOT.RooFit.Optimize(True), #optimize the treatment of constants in logL
                                 ROOT.RooFit.Offset(True), #set initial logL to 0
                                 ROOT.RooFit.Strategy(2), #internal code for the most precise minimization strategy of minuit2
                                 ROOT.RooFit.Hesse(True), #use Hesse to compute the uncertainty
                                 ROOT.RooFit.Save(True))


fit_result.Print("v")

c2 = ROOT.TCanvas("c2","fit",800,600)
frame = mass.frame()
dataset2.plotOn(frame)
peak_pdf_norm.plotOn(frame)
frame.Draw()
c2.Update()
c2.SaveAs("normalization_peaking_bkg_fit.pdf")
c2.Draw()

[#1] INFO:DataHandling -- RooAbsReal::attachToTree(mass) TTree Float_t branch mass will be converted to double precision.
[#0] WARNING:InputArguments -- The parameter 's1' with range [0, 0.5] of the RooGaussian 'g1' exceeds the safe range of (0, inf). Advise to limit its range.
[#0] WARNING:InputArguments -- The parameter 's2' with range [0, 0.5] of the RooGaussian 'g2' exceeds the safe range of (0, inf). Advise to limit its range.
[#0] WARNING:InputArguments -- The parameter 's3' with range [0, 0.5] of the RooGaussian 'g3' exceeds the safe range of (0, inf). Advise to limit its range.
[#1] INFO:Fitting -- RooAbsPdf::fitTo(peak_pdf_norm) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- Creation of NLL object took 808.483 μs
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_peak_pdf_norm_data2) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const opti

Info in <Minuit2>: MnSeedGenerator Computing seed using NumericalGradient calculator
Info in <Minuit2>: MnSeedGenerator Evaluated function and gradient in 45.8301 ms
Info in <Minuit2>: MnSeedGenerator Initial state: FCN =                 0 Edm =      0.1896446584 NCalls =     25
Warning in <Minuit2>: MnPosDef Matrix forced pos-def by adding to diagonal 0.00742921
Info in <Minuit2>: MnHesse Done after 88.1078 ms
Info in <Minuit2>: MnSeedGenerator run Hesse - Initial seeding state: 
  Minimum value : 0
  Edm           : 0.5706795733
  Internal parameters:	[                0    -0.4115168461    -0.1202898824     -1.168080485       -1.0758622    -0.9972832224]	
  Internal gradient  :	[      1.007017816     -6.508313351      506.6435871      15.01153695     -12.08400112      398.1365107]	
  Internal covariance matrix:
[[     0.77991891    -0.22609057  2.1800745e-05    0.021918711  -0.0082434427  -0.0059176975]
 [    -0.22609057     0.10391788  -1.071912e-05  -0.0039242541    0.010241687   0

2 c In addition to the peaking background we have the combinatorial background. Such background appear from a random combiantorial combinations of tracks. Knowing this, which sample will have a larger combinatorial contribution : $B_s^0 \to \mu^+ \mu^-$ or $B^+ \to J/\psi K^+$ ? 

In [15]:
## Now let's fit full fit 

dataset3 = ROOT.RooDataSet("data3", "", ROOT.RooArgList(mass), ROOT.RooFit.Import(norm_full))
dataset3.Print("v")

nbins = 500
mass.setBins(nbins)

datahist = ROOT.RooDataHist(
    "datahist", "",
    ROOT.RooArgSet(mass),
    dataset3
)

#First let's constrain the peaking background shape 

var = peak_pdf_norm.getParameters(dataset3).find("s1")
var.setConstant(True)

var = peak_pdf_norm.getParameters(dataset3).find("s2")
var.setConstant(True)

var = peak_pdf_norm.getParameters(dataset3).find("s3")
var.setConstant(True)

var = peak_pdf_norm.getParameters(dataset3).find("mu_peak")
var.setConstant(True)

var = peak_pdf_norm.getParameters(dataset3).find("f12")
var.setConstant(True)

var = peak_pdf_norm.getParameters(dataset3).find("f23")
var.setConstant(True)



#let's constrain the signal shape at least partially
var = signal_pdf_norm.getParameters(dataset3).find("f1")
var.setConstant(True)

var = signal_pdf_norm.getParameters(dataset3).find("SG1")
var.setConstant(True)

var = signal_pdf_norm.getParameters(dataset3).find("f")
var.setConstant(True)

var = signal_pdf_norm.getParameters(dataset3).find("alpha")
var.setConstant(True)

var = signal_pdf_norm.getParameters(dataset3).find("n")
var.setConstant(True)

var = signal_pdf_norm.getParameters(dataset3).find("SCB")
var.setConstant(True)

# let's leave the signal pdf floating

#this is the combinatorial component model
gamma_norm = ROOT.RooRealVar("gamma_norm", "gamma", -0.0001, -0.01, 0.01)

comb_pdf_norm = ROOT.RooExponential(
    "comb_pdf_norm","comb",
    mass,
    gamma_norm
)

#And we need to redefine the values of the yields, since the dataset change
N_ev_bu = ROOT.RooRealVar("N_ev_bu", "", 0.5 * dataset3.numEntries(), 0., 1.2 * dataset3.numEntries())
N_ev_peak = ROOT.RooRealVar("N_ev_peak", "", 150000.)
N_ev_comb = ROOT.RooRealVar("N_ev_comb", "", 0.4 * dataset3.numEntries(), 0., 1.2 * dataset3.numEntries())

#define extended pdf again with new yields
signal_pdf_norm.Print("v")
peak_pdf_norm.Print("v")
comb_pdf_norm.Print("v")

#total pdf
model = ROOT.RooAddPdf("model", "", ROOT.RooArgList(signal_pdf_norm, peak_pdf_norm, comb_pdf_norm), ROOT.RooArgList(N_ev_bu, N_ev_peak, N_ev_comb))
model.Print("v")
fit_result = model.fitTo(datahist,
                          ROOT.RooFit.Minimizer("Minuit2"),
                          ROOT.RooFit.Extended(True), #it is extended fit
                          ROOT.RooFit.Optimize(True), #optimize the treatment of constants in logL
                          ROOT.RooFit.Offset(True), #set initial logL to 0
                          ROOT.RooFit.Save(True))


fit_result.Print("v")

c3 = ROOT.TCanvas("c3","fit",800,600)
frame = mass.frame()
datahist.plotOn(frame)
model.plotOn(frame)
model.plotOn(frame, ROOT.RooFit.Components("peak_pdf_norm"), ROOT.RooFit.LineColor(ROOT.kGreen))
frame.Draw()
c3.Draw()
c3.SaveAs("normalization_fit.pdf")


[#1] INFO:DataHandling -- RooAbsReal::attachToTree(mass) TTree Float_t branch mass will be converted to double precision.
[#1] INFO:DataHandling -- RooTreeDataStore::loadValues(data3) Skipping event #5026713 because mass cannot accommodate the value 5.1
[#1] INFO:DataHandling -- RooTreeDataStore::loadValues(data3) Skipping event #6958425 because mass cannot accommodate the value 5.1
[#1] INFO:DataHandling -- RooTreeDataStore::loadValues(data3) Skipping event #16178251 because mass cannot accommodate the value 5.1
[#1] INFO:DataHandling -- RooTreeDataStore::loadValues(data3) Skipping event #16392283 because mass cannot accommodate the value 5.1
[#1] INFO:DataHandling -- RooTreeDataStore::loadValues(data3) Skipping ...
[#0] WARNING:DataHandling -- RooTreeDataStore::loadValues(data3) Ignored 16 out-of-range events
DataStore data3 ()
  Contains 53433984 entries
  Observables: 
    1)  mass = 5.15392  L(5.1 - 5.45)  "$m(J/#psi K^{+})$"
--- RooAbsArg ---
  Value State: DIRTY
  Shape State: D

Info in <Minuit2>: MnSeedGenerator Computing seed using NumericalGradient calculator
Info in <Minuit2>: MnSeedGenerator Evaluated function and gradient in 688.914 μs
Info in <Minuit2>: MnSeedGenerator Initial state: FCN =                 0 Edm =       14035996.09 NCalls =     21
Info in <Minuit2>: NegativeG2LineSearch Doing a NegativeG2LineSearch since one of the G2 component is negative
Info in <Minuit2>: NegativeG2LineSearch Done after 454.527 μs
Info in <Minuit2>: MnSeedGenerator Negative G2 found - new state: 
  Minimum value : -8.809742928
  Edm           : 14035044.36
  Internal parameters:	[    -0.1674480792    -0.3398369095    -0.2113687198     0.1118096827     -1.259582983]	
  Internal gradient  :	[      12725910.97     -22433330.36      39.55307392     -189940.8462     -53100609.32]	
  Internal covariance matrix:
[[  1.0981805e-07              0              0              0              0]
 [              0  3.4911256e-08              0              0              0]
 [     

2.d The dataset would corresponds to the same statistics as CMS $140~fb^{-1}$ dataset sued in https://linkinghub.elsevier.com/retrieve/pii/S0370269323002897. 
The CMS measure the branching ratio to be $3.83^{+0.378}_{-0.37}\textrm{(stat.)}{}^{+0.24}_{-0.21}\textrm{(syst.)}$.
LHCb best result is $3.09^{+0.46}_{-0.43}\textrm{(stat.)}{}^{+0.15}_{-0.11}\textrm{(syst.)}$ on just $9~fb^{-1}$. 

- How much more data LHCb needs to collect to match the statistical uncertainty of CMS? ($\sigma_{stat} \propto 1/\sqrt{N}$).
- Why CMS needs so much larger dataset than LHCb to match similar precision? 
- At the HL-LHC the dataset of CMS will be $3000~fb^{-1}$ and of LHCb $300~fb^{-1}$, assuming that the efficiencies do not change, who will win?  
- Can Belle-II contribute to the race?

2.e Assuming selection efficiency of $\varepsilon_{B^+\to J/\psi K^+} = 0.001$ and the $\varepsilon_{B^+\to\mu^+\mu^-} = 0.04$ compute the expected number of $B_s^0 \to \mu^+\mu^-$ events given the PDG BR value. Use $f_s/f_u = 0.256$. Why do you think the efficiency of $B^+\to J/\psi K^+$ is lower than of $B_s^0 \to \mu^+ \mu^-$